# Hands-on with Spark (Structured Streaming)

![spark](https://cdn-images-1.medium.com/max/300/1*c8CtvqKJDVUnMoPGujF5fA.png)

In the previous lesson, we learnt about Spark SQL, Dataframes and Pandas API. In this lesson, we will continue with the Structured Streaming.

Structured Streaming is a scalable and fault-tolerant stream processing engine built on the Spark SQL engine. You can express your streaming computation the same way you would express a batch computation on static data. The Spark SQL engine will take care of running it incrementally and continuously and updating the final result as streaming data continues to arrive.

You can use the Dataset/DataFrame API to express streaming aggregations, event-time windows, stream-to-batch joins, etc. The computation is executed on the same optimized Spark SQL engine. Finally, the system ensures end-to-end exactly-once fault-tolerance guarantees through checkpointing and Write-Ahead Logs. In short, Structured Streaming provides **fast, scalable, fault-tolerant, end-to-end exactly-once** stream processing without the user having to reason about streaming.

Internally, by default, Structured Streaming queries are processed using a micro-batch processing engine, which processes data streams as a series of small batch jobs thereby achieving end-to-end latencies as low as 100 milliseconds and exactly-once fault-tolerance guarantees.

## Installing and Initializing Spark

First, like previously, we'll need to install Spark and its dependencies:

1.   Java 8
2.   Apache Spark with Hadoop
3.   Findspark (used to locate the Spark in the system)


In [1]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4 pyspark-shell'

# set the options to connect to our Kafka cluster
options = {
    # "kafka.sasl.jaas.config": 'org.apache.kafka.common.security.scram.ScramLoginModule required username="YnJhdmUtZmlzaC0xMTQ2MyQSvwXBuLOQsV1W7YffuC8cDaZcA3fKQwakMhnQGgg" password="MDUxNjc4YzEtYzYxNy00NTE1LWEwNWYtMDBhODRlZmE0OGJm";',
    # "kafka.sasl.mechanism": "SCRAM-SHA-256",
    # "kafka.security.protocol" : "SASL_SSL",
    "kafka.bootstrap.servers": 'localhost:9092',
    "subscribe": 'pizza-orders',
}

In [ ]:
# # findspark is only required if you are using a standalone Spark installation (downloaded tar.gz)
# import findspark
# findspark.init()

The below cell may take a few minutes to run. It is slow because:
- Package downloading: Your PYSPARK_SUBMIT_ARGS includes a Kafka package that needs to be downloaded from Maven repositories on first run
- JVM cold start: Initial JVM startup is always slow
- Jupyter overhead: Running in a notebook adds some initialization overhead

In [2]:
# We will only use 2 cores below to speed up creation of the spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LearnSparkStreaming").master("local[2]").getOrCreate()

26/06/05 11:17:10 WARN Utils: Your hostname, Exclusives-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.20.10.11 instead (on interface en0)
26/06/05 11:17:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/exclusive/.ivy2/cache
The jars for the packages stored in: /Users/exclusive/.ivy2/jars


:: loading settings :: url = jar:file:/Users/exclusive/miniconda3/envs/kafka/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ecc0a50f-775c-4bac-96e0-9f097c071a1f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.4 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.4 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.4/spark-sql-kafka-0-10_2.12-3.5.4.jar ...
	[SUCCESSFUL ] org.apache.spark#spark-sql-

In [3]:
spark

# The above checks if spark is correctly installed and configured. If you see a spark session object, then you are good to go!

## Read and Analyze Kafka stream in "Batch" Mode

Let's start with reading and analyzing our `pizza-orders` kafka topic in the usual "batch" mode of Spark SQL and Dataframes. This is akin to the batch queries we did in the previous lesson. In this case we are taking the messages with the earliest to latest offsets of the topic as a single "batch".

In [4]:
pizza_df = spark.read.format('kafka')\
    .options(**options)\
    .load()

In [5]:
pizza_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
pizza_df.show()

26/06/05 11:19:21 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------------------+--------------------+------------+---------+------+--------------------+-------------+
|                 key|               value|       topic|partition|offset|           timestamp|timestampType|
+--------------------+--------------------+------------+---------+------+--------------------+-------------+
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     0|2026-06-05 11:07:...|            0|
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     1|2026-06-05 11:07:...|            0|
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     2|2026-06-05 11:07:...|            0|
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     3|2026-06-05 11:07:...|            0|
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     4|2026-06-05 11:07:...|            0|
|[7B 22 73 68 6F 7...|[7B 22 69 64 22 3...|pizza-orders|        0|     5|2026-06-05 11:09:...|            0|
|[7B 22 73 68 6F 7.

In [7]:
# Converting the key values to string for better readability

pizza_df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)").show()

26/06/05 11:20:15 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------------------+--------------------+
|                 key|               value|
+--------------------+--------------------+
|{"shop": "Luigis ...|{"id": 0, "shop":...|
|{"shop": "Ill Mak...|{"id": 1, "shop":...|
|{"shop": "Circula...|{"id": 2, "shop":...|
|{"shop": "Its-a m...|{"id": 3, "shop":...|
|{"shop": "Ill Mak...|{"id": 4, "shop":...|
|{"shop": "Ill Mak...|{"id": 0, "shop":...|
|{"shop": "Circula...|{"id": 1, "shop":...|
|{"shop": "Ill Mak...|{"id": 2, "shop":...|
|{"shop": "Ill Mak...|{"id": 3, "shop":...|
|{"shop": "Its-a m...|{"id": 4, "shop":...|
+--------------------+--------------------+



In [8]:
# From here on, no longer using Kafka specific code, just regular Spark DataFrame code

from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, StructType, ArrayType, StructField

In [9]:
pizza_schema = StructType([
  StructField("pizzaName", StringType()),
  StructField("additionalToppings", ArrayType(StringType())),
])

order_schema = StructType([
  StructField("address", StringType()),
  StructField("id", IntegerType()),
  StructField("name", StringType()),
  StructField("phoneNumber", StringType()),
  StructField("shop", StringType()),
  StructField("cost", DoubleType()),
  StructField("pizzas", ArrayType(pizza_schema)),
  StructField("timestamp", LongType()),
])

In [10]:
parsed_df = pizza_df.select("timestamp", from_json(col("value").cast("string"), order_schema).alias("value"))

In [11]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- address: string (nullable = true)
 |    |-- id: integer (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- phoneNumber: string (nullable = true)
 |    |-- shop: string (nullable = true)
 |    |-- cost: double (nullable = true)
 |    |-- pizzas: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- pizzaName: string (nullable = true)
 |    |    |    |-- additionalToppings: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |-- timestamp: long (nullable = true)



In [12]:
parsed_df.show(truncate=False)

26/06/05 11:22:19 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|timestamp              |value                                                                                                                                                                                                                                                                      |
+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2026-06-05 11:07:03.295|{113 Reeves Oval\nChanbury, UT 40179, 0, Katherine Wright, 922-262-0398x5836, Luigis Pizza, 3

We can use _dot notation_ to select the field within a `Struct`:

In [13]:
parsed_df.select("value.cost").show()

26/06/05 11:23:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-----+
| cost|
+-----+
|39.72|
| 0.71|
| 9.05|
|14.91|
|42.65|
| 1.46|
| 9.89|
| 8.58|
| 9.84|
|12.05|
+-----+



Computing the "total revenue" per shop:

In [14]:
parsed_df.groupBy("value.shop").sum("value.cost").show(truncate=False)

26/06/05 11:24:19 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+------------------------------------+-----------------------+
|shop                                |sum(value.cost AS cost)|
+------------------------------------+-----------------------+
|Ill Make You a Pizza You Cant Refuse|63.239999999999995     |
|Luigis Pizza                        |39.72                  |
|Its-a me! Mario Pizza!              |26.96                  |
|Circular Pi Pizzeria                |18.94                  |
+------------------------------------+-----------------------+



> 1. Count the no. of orders by shop.
> 2. Compute the avg revenue by shop and sort by highest to lowest.

In [15]:
from pyspark.sql.functions import min, max

In [16]:
parsed_df.select(min("timestamp"), max("timestamp")).show(truncate=False)

26/06/05 11:25:04 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-----------------------+-----------------------+
|min(timestamp)         |max(timestamp)         |
+-----------------------+-----------------------+
|2026-06-05 11:07:03.295|2026-06-05 11:09:18.235|
+-----------------------+-----------------------+



## Read and Analyze Kafka stream in "Streaming" Mode

The key idea in Structured Streaming is to treat a live data stream as a table that is being continuously appended. This leads to a new stream processing model that is very similar to a batch processing model. You will express your streaming computation as standard batch-like query as on a static table, and Spark runs it as an incremental query on the *unbounded input table*. Let’s understand this model in more detail.

Consider the input data stream as the “Input Table”. Every data item that is arriving on the stream is like a new row being appended to the Input Table.

![concept](https://spark.apache.org/docs/latest/img/structured-streaming-stream-as-a-table.png)

A query on the input will generate the “Result Table”. Every trigger interval (say, every 1 second), new rows get appended to the Input Table, which eventually updates the Result Table. Whenever the result table gets updated, we would want to write the changed result rows to an external sink.

![result table](https://spark.apache.org/docs/latest/img/structured-streaming-model.png)

To illustrate the use of this model, let’s understand the model in context of a word count model. The first lines DataFrame is the input table, and the final wordCounts DataFrame is the result table. Note that the query on streaming lines DataFrame to generate wordCounts is exactly the same as it would be a static DataFrame. However, when this query is started, Spark will continuously check for new data from the socket connection. If there is new data, Spark will run an “incremental” query that combines the previous running counts with the new data to compute updated counts, as shown below.

![example](https://spark.apache.org/docs/latest/img/structured-streaming-example-model.png)




Event-time is the time embedded in the data itself. For many applications, you may want to operate on this event-time.

For example, if you want to get the number of events generated by IoT devices every minute, then you probably want to use the time when the data was generated (that is, event-time in the data), rather than the time Spark receives them.

This event-time is very naturally expressed in this model – each event from the devices is a row in the table, and event-time is a column value in the row. This allows window-based aggregations (e.g. number of events every minute) to be just a special type of grouping and aggregation on the event-time column – each time window is a group and each row can belong to multiple windows/groups. Therefore, such event-time-window-based aggregation queries can be defined consistently on both a static dataset (e.g. from collected device events logs) as well as on a data stream, making the life of the user much easier.

Furthermore, this model naturally handles data that has arrived later than expected based on its event-time. Since Spark is updating the Result Table, it has full control over updating old aggregates when there is late data, as well as cleaning up old aggregates to limit the size of intermediate state data.

In [17]:
# Below is streaming, in contrast to read above, which is batch. The above read will read all the data in the topic and then stop, while the below read will keep running and processing new data as it arrives in the topic.
# The function is also different, where here is .readStream() and above is .read()

pizza_df = spark.readStream.format('kafka')\
    .options(**options)\
    .load()

In [18]:
pizza_df.isStreaming

True

In [19]:
pizza_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [20]:
parsed_df = pizza_df.select("timestamp", from_json(col("value").cast("string"), order_schema).alias("value"))

In [21]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- address: string (nullable = true)
 |    |-- id: integer (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- phoneNumber: string (nullable = true)
 |    |-- shop: string (nullable = true)
 |    |-- cost: double (nullable = true)
 |    |-- pizzas: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- pizzaName: string (nullable = true)
 |    |    |    |-- additionalToppings: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |-- timestamp: long (nullable = true)



In [22]:
query = parsed_df \
    .writeStream \
    .format("console") \
    .start()

# query.awaitTermination() # stops the script from exiting. But this is not required in Jupyter because the kernel stays alive. In real Spark applications, you need awaitTermination() to keep the job running.
# query.isActive
# query.recentProgress
# query.stop()

26/06/05 11:28:55 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/m6/dfw1bzjd4kl7gxgwhqjrfxf00000gn/T/temporary-af86dec2-451f-41e6-8b57-aefa0212d8a2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/05 11:28:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/05 11:28:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+--------------------+
|           timestamp|               value|
+--------------------+--------------------+
|2026-06-05 11:30:...|{69556 Price Lodg...|
+--------------------+--------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+--------------------+
|           timestamp|               value|
+--------------------+--------------------+
|2026-06-05 11:30:...|{284 Richard Gree...|
+--------------------+--------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+--------------------+--------------------+
|           timestamp|               value|
+--------------------+--------------------+
|2026-06-05 11:30:...|{4298 Schroeder C...|
+--------------------+--------------------+

-------------------------------------------
Ba

In [23]:
query.stop()

In [24]:
query.recentProgress

[{'id': 'b2ebce7e-9e2c-4c78-90b9-6397420cc817',
  'runId': '688b1883-6891-4919-91d5-91c0a47c8800',
  'name': None,
  'timestamp': '2026-06-05T03:28:55.643Z',
  'batchId': 0,
  'numInputRows': 0,
  'inputRowsPerSecond': 0.0,
  'processedRowsPerSecond': 0.0,
  'durationMs': {'addBatch': 67,
   'commitOffsets': 62,
   'getBatch': 12,
   'latestOffset': 146,
   'queryPlanning': 44,
   'triggerExecution': 379,
   'walCommit': 38},
  'stateOperators': [],
  'sources': [{'description': 'KafkaV2[Subscribe[pizza-orders]]',
    'startOffset': None,
    'endOffset': {'pizza-orders': {'0': 10}},
    'latestOffset': {'pizza-orders': {'0': 10}},
    'numInputRows': 0,
    'inputRowsPerSecond': 0.0,
    'processedRowsPerSecond': 0.0,
    'metrics': {'avgOffsetsBehindLatest': '0.0',
     'maxOffsetsBehindLatest': '0',
     'minOffsetsBehindLatest': '0'}}],
  'sink': {'description': 'org.apache.spark.sql.execution.streaming.ConsoleTable$@5dce2083',
   'numOutputRows': 0}},
 {'id': 'b2ebce7e-9e2c-4c78-9

### In the next few cells, we will provide a complete example of counting the number of orders per pizza shop, as the orders stream in.

In [25]:
# First check if any streams are active
spark.streams.active

[]

In [26]:
# If there are any active streams, stop them
for q in spark.streams.active:
    print(f"Stopping query: {q.name}")
    q.stop()

In [27]:
# Complete example with counting
pizza_df = spark.readStream.format('kafka')\
    .options(**options)\
    .load()

parsed_df = pizza_df.select("timestamp", from_json(col("value").cast("string"), order_schema).alias("value")) #.groupBy("value.shop").count()

shop_counts = parsed_df.groupBy("value.shop").count()

# outputMode("complete") rewrites all aggregated results every trigger — good for full snapshots.
# outputMode("update") - only updated rows are written
query = shop_counts \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

# The result will not be per order message, but rather an accumulation and showing the new status (by the batch number).

26/06/05 11:36:05 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/m6/dfw1bzjd4kl7gxgwhqjrfxf00000gn/T/temporary-cd069431-5049-48f9-9901-b5aa95c2ac45. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/05 11:36:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/05 11:36:05 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|shop|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------+-----+
|          shop|count|
+--------------+-----+
|Mammamia Pizza|    1|
+--------------+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-----+
|                shop|count|
+--------------------+-----+
|Ill Make You a Pi...|    1|
|      Mammamia Pizza|    1|
|Its-a me! Mario P...|    2|
|        Marios Pizza|    1|
+--------------------+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------------------+-----+
|                shop|count|
+--------------------+-----+
|Ill Make You a Pi...|    1|
|      Mammamia Pizza|    1|
|Its-a me! Mario P...|    3|
|        Marios Pizza|    1|
+--------------------+-----+



-------------------------------------------
Batch: 4
-------------------------------------------
+--------------------+-----+
|                shop|count|
+--------------------+-----+
|Ill Make You a Pi...|    1|
|      Mammamia Pizza|    3|
|Its-a me! Mario P...|    3|
|        Marios Pizza|    1|
+--------------------+-----+



-------------------------------------------
Batch: 5
-------------------------------------------
+--------------------+-----+
|                shop|count|
+--------------------+-----+
|Ill Make You a Pi...|    1|
|        Luigis Pizza|    1|
|      Mammamia Pizza|    3|
|Its-a me! Mario P...|    3|
|        Marios Pizza|    2|
+--------------------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/exclusive/miniconda3/envs/kafka/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/exclusive/miniconda3/envs/kafka/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/exclusive/miniconda3/envs/kafka/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [28]:
query.stop()

26/06/05 16:06:27 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 402619 ms exceeds timeout 120000 ms
26/06/05 16:06:27 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/05 16:51:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

You have come to the end of this exercise.

To delete the Kafka topic `pizza-orders`, use the command below in your terminal:

`./kafka-topics.sh --delete --topic pizza-orders --bootstrap-server localhost:9092`

To exit Kafka in your terminal, type `exit`.